In [4]:
pip install pandas numpy scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np
import os
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.metrics import Recall
from sklearn.utils.class_weight import compute_class_weight
import tensorflow.keras.backend as K


import warnings
warnings.filterwarnings('ignore')

seed = 9001
np.random.seed(seed)

In [6]:
INPUT_PATH = "/kaggle/input/datasets/thuhiuhong/traintestval-lstm"

X_train = np.load(os.path.join(INPUT_PATH, 'X_train (1).npy'))
y_train = np.load(os.path.join(INPUT_PATH, 'y_train (1).npy'))
X_val = np.load(os.path.join(INPUT_PATH, 'X_val (1).npy'))
y_val = np.load(os.path.join(INPUT_PATH, 'y_val (1).npy'))
X_test = np.load(os.path.join(INPUT_PATH, 'X_test (1).npy'))
y_test = np.load(os.path.join(INPUT_PATH, 'y_test (1).npy'))

In [7]:
X_train = np.asarray(X_train).astype(np.float32)
X_val   = np.asarray(X_val).astype(np.float32)
X_test  = np.asarray(X_test).astype(np.float32)

y_train = np.asarray(y_train).astype(np.int32).reshape(-1)
y_val   = np.asarray(y_val).astype(np.int32).reshape(-1)
y_test  = np.asarray(y_test).astype(np.int32).reshape(-1)

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("X_val:  ", X_val.shape, X_val.dtype)
print("y_val:  ", y_val.shape, y_val.dtype)
print("X_test: ", X_test.shape, X_test.dtype)
print("y_test: ", y_test.shape, y_test.dtype)

assert X_train.ndim == 3, f"X_train phải là 3D, hiện tại là {X_train.ndim}D"
assert X_val.ndim == 3, f"X_val phải là 3D, hiện tại là {X_val.ndim}D"
assert X_test.ndim == 3, f"X_test phải là 3D, hiện tại là {X_test.ndim}D"

assert len(X_train) == len(y_train), "X_train và y_train lệch số mẫu"
assert len(X_val) == len(y_val), "X_val và y_val lệch số mẫu"
assert len(X_test) == len(y_test), "X_test và y_test lệch số mẫu"

print("timesteps =", X_train.shape[1])
print("n_features =", X_train.shape[2])
print("Train class distribution:", np.bincount(y_train))
print("Val class distribution:", np.bincount(y_val))
print("Test class distribution:", np.bincount(y_test))

X_train: (145772, 10, 133) float32
y_train: (145772,) int32
X_val:   (190797, 10, 133) float32
y_val:   (190797,) int32
X_test:  (238447, 10, 133) float32
y_test:  (238447,) int32
timesteps = 10
n_features = 133
Train class distribution: [134796  10976]
Val class distribution: [187169   3628]
Test class distribution: [234296   4151]


In [8]:
import sys
MODULE_PATH = "/kaggle/input/datasets/thuhiuhong/lstm-utils"
if MODULE_PATH not in sys.path:
    sys.path.append(MODULE_PATH)

from model_utils import (
    create_bilstm,
    get_callbacks,
    find_best_threshold,
    full_evaluation,
)

In [9]:
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = {
    int(cls): float(w) for cls, w in zip(np.unique(y_train), class_weights_array)
}
print("Trọng số lớp Train:", class_weights_dict)

lstm_units_options = [32, 64, 80]
dropout_rate_options = [0.3, 0.4, 0.5]
batch_size_options = [256, 512]
num_epochs = 15

results = []

for n_units in lstm_units_options:
    for dropout_rate in dropout_rate_options:
        for batch_size in batch_size_options:
            print(f"\nĐang test: units={n_units}, dropout={dropout_rate}, batch_size={batch_size}")

            K.clear_session()

            model = create_bilstm(
                n_units=n_units,
                dropout=dropout_rate,
                seq_len=X_train.shape[1],
                n_features=X_train.shape[2],
                lr=1e-3
            )

            history = model.fit(
                X_train, y_train,
                epochs=num_epochs,
                batch_size=batch_size,
                class_weight=class_weights_dict,
                validation_data=(X_val, y_val),
                shuffle=True,
                verbose=0
            )

            val_loss, val_accuracy, val_auroc, val_auprc, val_recall, val_precision = model.evaluate(
                X_val, y_val, verbose=0
            )

            print(
                f"Val AUROC: {val_auroc:.4f} | "
                f"Val AUPRC: {val_auprc:.4f} | "
                f"Recall: {val_recall:.4f} | "
                f"Precision: {val_precision:.4f}"
            )

            results.append((
                n_units, dropout_rate, batch_size,
                val_loss, val_accuracy, val_auroc, val_auprc, val_recall, val_precision
            ))

# Chọn theo AUPRC thay vì AUROC
best_result = sorted(results, key=lambda x: x[5], reverse=True)[0]

print("\n" + "="*60)
print("Best Hyperparameters:")
print(f"- LSTM Units:   {best_result[0]}")
print(f"- Dropout Rate: {best_result[1]}")
print(f"- Batch Size:   {best_result[2]}")
print(f"- Val AUROC:    {best_result[5]:.4f}")
print(f"- Val AUPRC:    {best_result[6]:.4f}")
print("="*60)

Trọng số lớp Train: {0: 0.5407133742841034, 1: 6.64048833819242}

Đang test: units=16, dropout=0.3, batch_size=256


I0000 00:00:1778323305.923917      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778323305.929850      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1778323314.606582     165 cuda_dnn.cc:529] Loaded cuDNN version 91002


Val AUROC: 0.7670 | Val AUPRC: 0.0562 | Recall: 0.6122 | Precision: 0.0586

Đang test: units=16, dropout=0.3, batch_size=512
Val AUROC: 0.7637 | Val AUPRC: 0.0608 | Recall: 0.5907 | Precision: 0.0649

Đang test: units=16, dropout=0.4, batch_size=256
Val AUROC: 0.7586 | Val AUPRC: 0.0813 | Recall: 0.4898 | Precision: 0.0793

Đang test: units=16, dropout=0.4, batch_size=512
Val AUROC: 0.7733 | Val AUPRC: 0.0676 | Recall: 0.5703 | Precision: 0.0696

Đang test: units=16, dropout=0.5, batch_size=256
Val AUROC: 0.7755 | Val AUPRC: 0.0675 | Recall: 0.5921 | Precision: 0.0682

Đang test: units=16, dropout=0.5, batch_size=512
Val AUROC: 0.7637 | Val AUPRC: 0.0624 | Recall: 0.5973 | Precision: 0.0624

Đang test: units=32, dropout=0.3, batch_size=256
Val AUROC: 0.7199 | Val AUPRC: 0.0634 | Recall: 0.4300 | Precision: 0.0772

Đang test: units=32, dropout=0.3, batch_size=512
Val AUROC: 0.7429 | Val AUPRC: 0.0573 | Recall: 0.4859 | Precision: 0.0701

Đang test: units=32, dropout=0.4, batch_size=256


In [10]:
K.clear_session()

final_model = create_bilstm(
    n_units=best_result[0],
    dropout=best_result[1],
    seq_len=X_train.shape[1],
    n_features=X_train.shape[2],
    lr=1e-3
)

callbacks = get_callbacks(
    checkpoint_path='best_lstm_model.keras',
    monitor='val_auroc'
)

history = final_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=best_result[2],
    class_weight=class_weights_dict,
    validation_data=(X_val, y_val),
    callbacks=callbacks,
    shuffle=True,
    verbose=1
)

Epoch 1/50
568/570 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5852 - auprc: 0.1650 - auroc: 0.6933 - loss: 0.6408 - precision: 0.1238 - recall: 0.7035
Epoch 1: val_auroc improved from -inf to 0.82210, saving model to best_lstm_model.keras
570/570 ━━━━━━━━━━━━━━━━━━━━ 20s 27ms/step - accuracy: 0.5857 - auprc: 0.1653 - auroc: 0.6936 - loss: 0.6405 - precision: 0.1240 - recall: 0.7036 - val_accuracy: 0.7608 - val_auprc: 0.0929 - val_auroc: 0.8221 - val_loss: 0.4854 - val_precision: 0.0570 - val_recall: 0.7456 - learning_rate: 0.0010
Epoch 2/50
569/570 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7623 - auprc: 0.2987 - auroc: 0.8401 - loss: 0.5015 - precision: 0.2126 - recall: 0.7916
Epoch 2: val_auroc improved from 0.82210 to 0.82482, saving model to best_lstm_model.keras
570/570 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.7623 - auprc: 0.2987 - auroc: 0.8401 - loss: 0.5014 - precision: 0.2126 - recall: 0.7917 - val_accuracy: 0.7123 - val_auprc: 0.0968 - val_auroc: 0.8248 - v

In [11]:
val_prob = final_model.predict(X_val, verbose=0).ravel()

best_val_threshold = find_best_threshold(
    y_true=y_val,
    y_prob=val_prob,
    min_sensitivity=0.80,
    threshold_range=(0.05, 0.95),
    step=0.01
)

print("Best threshold from validation:")
print(best_val_threshold)

Best threshold from validation:
{'threshold': 0.47, 'accuracy': 0.6970497439687207, 'sensitivity': np.float64(0.8065049614112458), 'specificity': np.float64(0.6949281130956515), 'precision': 0.0487455435977743, 'recall': 0.8065049614112458, 'f1': 0.09193452100417884, 'youden_j': np.float64(0.5014330745068973), 'tn': 130069, 'fp': 57100, 'fn': 702, 'tp': 2926}


In [12]:
test_prob = final_model.predict(X_test, verbose=0).ravel()

full_evaluation(
    y_true=y_test,
    y_prob=test_prob,
    threshold=best_val_threshold["threshold"],
    label="TEST"
)

df_lstm = pd.DataFrame({
    'y_true': y_test,
    'pred_lstm': test_prob,
    'pred_label': (test_prob >= best_val_threshold["threshold"]).astype(int)
})


  TEST — threshold = 0.47
  AUROC        : 0.8416
  AUPRC        : 0.1028
  Accuracy     : 0.7045
  Sensitivity  : 0.8215
  Specificity  : 0.7024
  Precision    : 0.0466
  Recall       : 0.8215
  F1-score     : 0.0882
  Youden's J   : 0.5239
  Confusion    : TN=164569 FP=69727 FN=741 TP=3410

